[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C13_RL_Foundations_Course/02_q_learning_td/02_q_learning_td.ipynb)

# 02 · 时序差分与 Q-learning（纯 numpy）

目标：从零实现 **TD(0)**、**Q-learning(off-policy)**、**SARSA(on-policy)**、**ε-greedy**，在 GridWorld 上学到最优、对拍 DP 真值，在 **Cliff Walking** 上看清 on/off-policy 的行为差异。

路线：可采样 GridWorld + DP 真值 → TD(0) 预测对拍解析 V^π → ε-greedy → Q-learning 学到最优 → SARSA → Cliff Walking 对比 → ✏️ 练习 → 📖 答案 → 🧪 收敛性胶囊。

> 心智模型：**TD 误差 δ = r + γV(s') − V(s) = 惊喜程度**。Q-learning 目标用 **max**(学 Q*，off-policy)，SARSA 用**实际动作**(学当前策略，on-policy)。一字之差，行为天壤。

## 1 · 可采样的 GridWorld + DP 真值（对拍基准）

和模块 01 同一个随机 GridWorld，但现在 agent **不知道** (P,R)，只能 `step()` 采样一条 (s,a,r,s',done)。

我们仍在幕后用 DP 算出 $Q^*, V^*$ 作为**对拍真值**——这是表格环境的奢侈：能验证无模型算法学得对不对。

In [ ]:
import numpy as np

class GridWorld:
    def __init__(self, gamma=0.9, slip=0.2, step_reward=-0.04):
        self.n_rows, self.n_cols = 3, 4
        self.obstacles = {(1, 1)}
        self.terminals = {(0, 3): 1.0, (1, 3): -1.0}
        self.step_reward = step_reward; self.gamma = gamma; self.slip = slip
        self.moves = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}
        self.perp  = {0: (3, 1), 1: (0, 2), 2: (1, 3), 3: (2, 0)}
        self.states = [(r, c) for r in range(self.n_rows) for c in range(self.n_cols)
                       if (r, c) not in self.obstacles]
        self.S, self.A = len(self.states), 4
        self.sidx = {s: i for i, s in enumerate(self.states)}
        self.start = (2, 0)
        self.nonterm = [s for s in self.states if s not in self.terminals]
    def is_terminal(self, s): return s in self.terminals
    def _move(self, s, a):
        dr, dc = self.moves[a]; r, c = s; nr, nc = r + dr, c + dc
        if (nr, nc) in self.obstacles or not (0 <= nr < self.n_rows and 0 <= nc < self.n_cols):
            return s
        return (nr, nc)
    def reset(self, rng=None, exploring=False):
        self.pos = self.nonterm[rng.integers(0, len(self.nonterm))] if (exploring and rng is not None) else self.start
        return self.pos
    def step(self, a, rng):
        u = rng.random()                       # 采样滑动
        if u < 1 - self.slip:                  act = a
        elif u < 1 - self.slip + self.slip / 2: act = self.perp[a][0]
        else:                                  act = self.perp[a][1]
        sp = self._move(self.pos, act)
        r = self.terminals[sp] if sp in self.terminals else self.step_reward
        done = sp in self.terminals
        self.pos = sp
        return sp, r, done

def dp_solve(env):
    '''幕后用模型解 Q*,V*,π*（对拍真值）。'''
    S, A = env.S, env.A
    P = np.zeros((S, A, S)); R = np.zeros((S, A))
    for s in env.states:
        i = env.sidx[s]
        for a in range(A):
            if env.is_terminal(s): P[i, a, i] += 1.0; continue
            out = {}; la, ra = env.perp[a]
            for prob, ac in [(1 - env.slip, a), (env.slip / 2, la), (env.slip / 2, ra)]:
                sp = env._move(s, ac); out[sp] = out.get(sp, 0.0) + prob
            for sp, prob in out.items():
                j = env.sidx[sp]; rr = env.terminals[sp] if sp in env.terminals else env.step_reward
                P[i, a, j] += prob; R[i, a] += prob * rr
    tm = np.array([env.is_terminal(s) for s in env.states])
    V = np.zeros(S)
    for _ in range(2000):
        Q = R + env.gamma * (P @ V); Vn = Q.max(1); Vn[tm] = 0.0
        if np.max(np.abs(Vn - V)) < 1e-13: break
        V = Vn
    Qs = R + env.gamma * (P @ V)
    return Qs, V, Qs.argmax(1), P, R, tm

env = GridWorld()
Q_star, V_star, pi_star, P, R, tm = dp_solve(env)
print('DP 真值已备好。V* (左下起点 (2,0)) =', round(V_star[env.sidx[(2,0)]], 4))
# 验证采样环境与模型一致：从 (2,0) 选「上」大量采样，频率≈0.8 到 (1,0)
rng = np.random.default_rng(0)
counts = {}
for _ in range(20000):
    env.reset(); env.pos = (2, 0)
    sp, r, d = env.step(0, rng); counts[sp] = counts.get(sp, 0) + 1
freq_up = counts[(1, 0)] / 20000
print(f'从 (2,0) 选「上」落到 (1,0) 的采样频率 = {freq_up:.3f} (理论 0.8)')
assert abs(freq_up - 0.8) < 0.02, '采样环境应与模型一致'
print('✅ 可采样 GridWorld 就绪，DP 真值备好用于对拍')

## 2 · TD(0) 预测：估计固定策略的 V^π

给定固定策略（这里用 DP 的最优策略 π*），用 TD(0) 更新 $V(s) \leftarrow V(s) + \alpha[r + \gamma V(s') - V(s)]$ 估其价值。

**无需模型**，只靠采样。它应收敛到该策略的解析价值 $V^\pi$（在被访问到的状态上）。注意终止态目标不带 γV。

In [ ]:
def td0_predict(env, policy, n_episodes=50000, alpha=0.05, seed=1, max_steps=100, exploring=True):
    rng = np.random.default_rng(seed)
    V = np.zeros(env.S)
    for ep in range(n_episodes):
        s = env.reset(rng, exploring=exploring); si = env.sidx[s]
        for _ in range(max_steps):
            a = int(policy[si])
            sp, r, done = env.step(a, rng); spi = env.sidx[sp]
            target = r + (0.0 if done else env.gamma * V[spi])   # 终止态不自举!
            V[si] += alpha * (target - V[si])
            si = spi
            if done: break
    V[tm] = 0.0
    return V

# 解析地评估 π*（对拍基准）
def policy_eval_analytic(policy, P, R, gamma, tm):
    S = P.shape[0]; Ppi = P[np.arange(S), policy]; Rpi = R[np.arange(S), policy]
    V = np.linalg.solve(np.eye(S) - gamma * Ppi, Rpi); V[tm] = 0.0
    return V

V_analytic = policy_eval_analytic(pi_star, P, R, env.gamma, tm)
V_td = td0_predict(env, pi_star, n_episodes=60000, alpha=0.05, seed=1, exploring=True)
non_term = ~tm
abs_err = np.abs(V_td[non_term] - V_analytic[non_term])
max_err, mean_err = abs_err.max(), abs_err.mean()
print('TD(0) 估计 vs 解析 V^π：')
for s in [(2, 0), (0, 0), (2, 3)]:
    i = env.sidx[s]
    print(f'  {s}: TD={V_td[i]:+.3f}  解析={V_analytic[i]:+.3f}')
print(f'平均误差 = {mean_err:.3f},  最大误差 = {max_err:.3f}')
# 平均误差小 -> 整体学准；注意常数 α 的 TD 不收敛到「点」而是在真值附近「悬停」
# (回忆收敛性那节的 Robbins-Monro：常数 α 不满足 Σα²<∞)，故高方差状态(紧邻陷阱的)悬停幅度较大
assert mean_err < 0.05, 'TD(0) 整体应逼近解析 V^π（平均误差小）'
assert max_err < 0.16, '无状态应严重偏离（高方差态因常数α悬停幅度稍大是正常的）'
print('✅ TD(0) 无模型预测逼近解析精确价值（平均误差<0.05）—— bootstrapping + 采样有效')
print('   注：常数 α 使 TD 在真值附近悬停而非精确收敛(Robbins-Monro)，紧邻陷阱的高方差态悬停更明显')

## 3 · ε-greedy：探索与利用的最简平衡

以 $1-\varepsilon$ 选当前最优动作（利用），$\varepsilon$ 随机探索。先把这个工具写出来并验证其统计行为。

In [ ]:
def epsilon_greedy(Q_row, eps, rng, A):
    if rng.random() < eps:
        return int(rng.integers(0, A))               # 探索：随机
    return int(rng.choice(np.flatnonzero(Q_row == Q_row.max())))  # 利用：贪心(平局随机)

rng = np.random.default_rng(0)
Q_row = np.array([0.1, 0.9, 0.2, 0.3])   # 动作 1 最优
N = 100000
picks = np.array([epsilon_greedy(Q_row, eps=0.1, rng=rng, A=4) for _ in range(N)])
freq = np.bincount(picks, minlength=4) / N
print('eps=0.1 时各动作被选频率:', np.round(freq, 3))
# 最优动作1: 利用(0.9) + 探索时也可能选中(0.1/4) = 0.925
assert abs(freq[1] - (0.9 + 0.1 / 4)) < 0.01, '最优动作频率应≈1-ε+ε/A'
# 非最优动作: 仅探索选中 = 0.1/4 = 0.025
assert abs(freq[0] - 0.1 / 4) < 0.01, '非最优动作频率应≈ε/A'
print('✅ ε-greedy 正确：1-ε 利用最优，ε 均匀探索')

## 4 · Q-learning：off-policy 控制，学到最优

目标用 **max**：$Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$。

用 ε-greedy 探索（ε 衰减）+ exploring starts（随机起点，保证充分探索）。学到的贪心策略应**达到最优价值**（对拍 DP 的 V*）。

In [ ]:
def q_learning(env, n_episodes=40000, alpha=0.2, eps0=1.0, eps_min=0.1,
               seed=0, max_steps=50, exploring=True):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.S, env.A))
    for ep in range(n_episodes):
        eps = max(eps_min, eps0 * (1 - ep / n_episodes))    # 线性衰减
        s = env.reset(rng, exploring=exploring); si = env.sidx[s]
        for _ in range(max_steps):
            a = epsilon_greedy(Q[si], eps, rng, env.A)       # 行为策略(探索)
            sp, r, done = env.step(a, rng); spi = env.sidx[sp]
            target = r + (0.0 if done else env.gamma * Q[spi].max())  # 目标=贪心(max!)
            Q[si, a] += alpha * (target - Q[si, a])
            si = spi
            if done: break
    return Q

Q_ql = q_learning(env, n_episodes=40000, alpha=0.2, eps_min=0.1, seed=0)
pi_ql = Q_ql.argmax(1)
arrows = {0: '^', 1: '>', 2: 'v', 3: '<'}
def show_policy(pi):
    for r in range(env.n_rows):
        row = []
        for c in range(env.n_cols):
            s = (r, c)
            if s in env.obstacles: row.append('X')
            elif s in env.terminals: row.append('*')
            else: row.append(arrows[pi[env.sidx[s]]])
        print('  ' + ' '.join(row))
print('Q-learning 学到的策略:'); show_policy(pi_ql)
print('DP 最优策略:'); show_policy(pi_star)

# 稳健正确性判据：用真模型评估「学到的策略」，其价值应≈V*（即便个别平局态动作不同）
V_learned = policy_eval_analytic(pi_ql, P, R, env.gamma, tm)
val_gap = np.max(np.abs(V_learned[non_term] - V_star[non_term]))
n_match = np.sum(pi_ql[non_term] == pi_star[non_term])
print(f'\n学到策略的价值 vs V* 最大差 = {val_gap:.3f}  | 策略匹配 {n_match}/{non_term.sum()} 态')
assert val_gap < 0.06, '学到的策略价值应≈最优(个别平局态可不同但价值几乎不损)'
assert n_match >= 7, '绝大多数状态应学到最优动作'
print('✅ Q-learning 无模型学到(近乎)最优策略 —— 其价值对拍 DP 的 V*')

## 5 · Q 表本身对拍 Q*（被充分访问的状态）

不只策略对，学到的 $Q$ 值本身也应逼近 $Q^*$。我们看几个被频繁访问的状态上 Q 表与 DP 真值的差距。

In [ ]:
# 在常被访问的状态上对拍 Q 表（远角/罕访状态 Q 估计较糙是正常的）
key_states = [(2, 0), (1, 0), (0, 0), (0, 1), (0, 2)]
print(f"{'状态':>8} {'最优动作':>8} {'Q_learned(最优a)':>16} {'Q*(最优a)':>12}")
for s in key_states:
    i = env.sidx[s]; a = pi_star[i]
    print(f'{str(s):>8} {arrows[a]:>8} {Q_ql[i, a]:>16.3f} {Q_star[i, a]:>12.3f}')
errs = [abs(Q_ql[env.sidx[s], pi_star[env.sidx[s]]] - Q_star[env.sidx[s], pi_star[env.sidx[s]]])
        for s in key_states]
print(f'关键状态最优动作的 Q 值最大误差 = {max(errs):.3f}')
assert max(errs) < 0.1, '常访问状态的 Q 值应逼近 Q*'
print('✅ Q 表在被充分访问的状态上逼近 Q*（不只策略对，价值也对）')

## 6 · SARSA：on-policy 控制

把 Q-learning 目标里的 `max` 换成**实际选的下一动作** a'：$Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma Q(s',a') - Q(s,a)]$。

它学的是「当前 ε-greedy 策略」的价值（on-policy）。在 GridWorld 这种探索无大碍的环境，它也能学到近乎最优的策略。

In [ ]:
def sarsa(env, n_episodes=40000, alpha=0.2, eps0=1.0, eps_min=0.1,
          seed=0, max_steps=50, exploring=True):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.S, env.A))
    for ep in range(n_episodes):
        eps = max(eps_min, eps0 * (1 - ep / n_episodes))
        s = env.reset(rng, exploring=exploring); si = env.sidx[s]
        a = epsilon_greedy(Q[si], eps, rng, env.A)
        for _ in range(max_steps):
            sp, r, done = env.step(a, rng); spi = env.sidx[sp]
            ap = epsilon_greedy(Q[spi], eps, rng, env.A)      # 实际选的下一动作
            target = r + (0.0 if done else env.gamma * Q[spi, ap])  # 用 a' 而非 max!
            Q[si, a] += alpha * (target - Q[si, a])
            si, a = spi, ap
            if done: break
    return Q

Q_sa = sarsa(env, n_episodes=40000, alpha=0.2, eps_min=0.1, seed=0)
pi_sa = Q_sa.argmax(1)
V_sa = policy_eval_analytic(pi_sa, P, R, env.gamma, tm)
print('SARSA 学到的策略:'); show_policy(pi_sa)
gap_sa = np.max(np.abs(V_sa[non_term] - V_star[non_term]))
print(f'SARSA 策略价值 vs V* 最大差 = {gap_sa:.3f}')
assert gap_sa < 0.1, 'SARSA 也应学到接近最优(GridWorld 探索代价小)'
print('✅ SARSA(on-policy) 在 GridWorld 上学到接近最优策略')

## 7 · Cliff Walking：on/off-policy 的行为分水岭

经典实验（Sutton & Barto 例 6.6）：4×12 网格，左下起点、右下目标，底边中间是**悬崖**(掉下 −100 并送回起点)，每步 −1，γ=1。

最短路紧贴悬崖(最优但危险)，安全路绕顶行(远但稳)。固定 ε=0.1（不衰减）训练，比较**在线回报**：
- **Q-learning** 学最优(贴崖)路，但 ε 探索常把它推下崖 → 在线回报更低、更抖。
- **SARSA** 把「我还会探索」算进价值，学更安全的绕路 → 在线回报更高。

In [ ]:
class CliffWalking:
    def __init__(self, gamma=1.0):
        self.rows, self.cols = 4, 12; self.gamma = gamma
        self.start, self.goal = (3, 0), (3, 11)
        self.cliff = set((3, c) for c in range(1, 11))
        self.moves = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}
        self.states = [(r, c) for r in range(self.rows) for c in range(self.cols)]
        self.S, self.A = self.rows * self.cols, 4
        self.sidx = {s: i for i, s in enumerate(self.states)}
    def reset(self): self.pos = self.start; return self.pos
    def step(self, a, rng):
        dr, dc = self.moves[a]; r, c = self.pos
        nr, nc = min(max(r + dr, 0), self.rows - 1), min(max(c + dc, 0), self.cols - 1)
        np_ = (nr, nc)
        if np_ in self.cliff:
            self.pos = self.start; return self.start, -100.0, False  # 掉崖：重罚+回起点
        self.pos = np_
        return np_, -1.0, (np_ == self.goal)

def run_cliff(env, algo, n_episodes=2000, alpha=0.5, eps=0.1, seed=0, max_steps=400):
    rng = np.random.default_rng(seed); Q = np.zeros((env.S, env.A)); returns = []
    for ep in range(n_episodes):
        s = env.reset(); si = env.sidx[s]; total = 0.0
        a = epsilon_greedy(Q[si], eps, rng, env.A)
        for _ in range(max_steps):
            sp, r, done = env.step(a, rng); spi = env.sidx[sp]; total += r
            ap = epsilon_greedy(Q[spi], eps, rng, env.A)
            if algo == 'q':
                target = r + (0.0 if done else env.gamma * Q[spi].max())   # off-policy
            else:
                target = r + (0.0 if done else env.gamma * Q[spi, ap])     # on-policy
            Q[si, a] += alpha * (target - Q[si, a])
            si, a = spi, ap
            if done: break
        returns.append(total)
    return Q, np.array(returns)

cliff = CliffWalking()
Q_q, ret_q = run_cliff(cliff, 'q', seed=0)
Q_s, ret_s = run_cliff(cliff, 'sarsa', seed=0)
avg_q, avg_s = ret_q[-200:].mean(), ret_s[-200:].mean()
print(f'Q-learning 末期平均在线回报 = {avg_q:.1f}')
print(f'SARSA      末期平均在线回报 = {avg_s:.1f}')
# 经典结论：SARSA 在线回报更高(走安全路，少掉崖)
assert avg_s > avg_q, 'SARSA 的在线回报应高于 Q-learning(安全路)'
print('✅ Cliff Walking 经典结论：SARSA(on-policy)走安全路、在线回报更高；')
print('   Q-learning(off-policy)学最优路但探索中常掉崖、在线回报更低。')

**对比 Q-learning 与 SARSA 学到的路径**（贪心轨迹）。Q-learning 紧贴悬崖（行 2 即悬崖上方），SARSA 绕到更靠上的安全行。

In [ ]:
def greedy_path(env, Q, max_steps=30):
    s = env.reset(); path = [s]
    for _ in range(max_steps):
        a = int(Q[env.sidx[s]].argmax())
        dr, dc = env.moves[a]; r, c = s
        nr, nc = min(max(r + dr, 0), env.rows - 1), min(max(c + dc, 0), env.cols - 1)
        s = (nr, nc)
        if s in env.cliff: s = env.start
        path.append(s)
        if s == env.goal: break
    return path

path_q = greedy_path(cliff, Q_q)
path_s = greedy_path(cliff, Q_s)
min_row_q = min(r for r, c in path_q)   # 走到的最高行(越小越靠上=越安全)
min_row_s = min(r for r, c in path_s)
print(f'Q-learning 贪心路最高到达行 {min_row_q} (贴崖, 行越大越靠近悬崖行3)')
print(f'SARSA      贪心路最高到达行 {min_row_s} (绕远更靠上更安全)')
assert path_q[-1] == cliff.goal and path_s[-1] == cliff.goal, '两者贪心路都应到达目标'
assert min_row_s <= min_row_q, 'SARSA 的安全路应绕到更靠上的行'
print('✅ Q-learning 贴悬崖(最优), SARSA 绕安全路 —— 一个 max 之差的行为后果')

---
## ✏️ 练习 1：TD 误差与单步 TD 更新

实现两个原子操作：
- `td_error(V, s, r, sp, done, gamma)`：返回 $\delta = r + (1-\text{done})\gamma V(s') - V(s)$（注意终止态处理）；
- `td_update(V, s, delta, alpha)`：原地更新 `V[s] += alpha*delta` 并返回 V。

In [ ]:
def td_error(V, s, r, sp, done, gamma):
    # TODO: 返回 TD 误差；done=True 时目标不含 γV(s')
    raise NotImplementedError

def td_update(V, s, delta, alpha):
    # TODO: V[s] += alpha*delta; return V
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
V = np.array([0.0, 1.0, 2.0])
# 非终止: δ = 0.5 + 0.9*2.0 - 0.0 = 2.3
d = td_error(V, 0, r=0.5, sp=2, done=False, gamma=0.9)
assert abs(d - 2.3) < 1e-9, f'非终止 TD 误差应为 2.3, 得 {d}'
# 终止: δ = 1.0 + 0 - 1.0 = 0.0 (目标不自举)
d2 = td_error(V, 1, r=1.0, sp=2, done=True, gamma=0.9)
assert abs(d2 - 0.0) < 1e-9, f'终止态 TD 误差应为 0.0, 得 {d2}'
V = td_update(V.copy(), 0, delta=2.3, alpha=0.1)
assert abs(V[0] - 0.23) < 1e-9, 'TD 更新应为 V[0]=0.23'
print('✅ 练习 1 通过：TD 误差(含终止态处理) 与单步更新正确')

## ✏️ 练习 2：把 Q-learning 改成 SARSA

下面给出 Q-learning 的目标计算。实现 `sarsa_target(Q, r, spi, ap, done, gamma)`：
用**实际下一动作** `ap` 而非 max，返回 SARSA 目标 $r + (1-\text{done})\gamma Q[s',a']$。

再实现 `q_target(Q, r, spi, done, gamma)` 作对照（用 max）。比较两者在同一 (s',Q) 上的差异。

In [ ]:
def q_target(Q, r, spi, done, gamma):
    # TODO: Q-learning 目标，用 max_a' Q[spi,a']
    raise NotImplementedError

def sarsa_target(Q, r, spi, ap, done, gamma):
    # TODO: SARSA 目标，用实际动作 Q[spi, ap]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
Q = np.array([[0.0, 0.0], [1.0, 5.0]])   # 状态1: 动作0=1, 动作1=5(更优)
# Q-learning 目标用 max=5: 1 + 0.9*5 = 5.5
qt = q_target(Q, r=1.0, spi=1, done=False, gamma=0.9)
assert abs(qt - 5.5) < 1e-9, f'Q 目标应 5.5, 得 {qt}'
# SARSA 若实际选了次优动作 0 (值=1): 1 + 0.9*1 = 1.9
st = sarsa_target(Q, r=1.0, spi=1, ap=0, done=False, gamma=0.9)
assert abs(st - 1.9) < 1e-9, f'SARSA(选次优动作) 目标应 1.9, 得 {st}'
# 终止态: 两者都=r
assert abs(q_target(Q, 2.0, 1, True, 0.9) - 2.0) < 1e-9
print('✅ 练习 2 通过：Q-learning 用 max(乐观)，SARSA 用实际动作(随当前策略)')

## ✏️ 练习 3：ε 衰减调度

实现两种 ε 衰减：
- `eps_linear(ep, n_eps, eps0, eps_min)`：线性从 eps0 降到 eps_min；
- `eps_exponential(ep, decay, eps0, eps_min)`：指数 `max(eps_min, eps0 * decay**ep)`。

好的调度应：早期 ε 大(多探索)、晚期 ε 小(多利用)、且不低于 eps_min(永远留一点探索)。

In [ ]:
def eps_linear(ep, n_eps, eps0=1.0, eps_min=0.05):
    # TODO: 线性插值 eps0 -> eps_min，再 max(eps_min, ...)
    raise NotImplementedError

def eps_exponential(ep, decay=0.999, eps0=1.0, eps_min=0.05):
    # TODO: max(eps_min, eps0 * decay**ep)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(eps_linear(0, 1000) - 1.0) < 1e-9, '第0回合应=eps0'
assert abs(eps_linear(1000, 1000) - 0.05) < 1e-9, '末回合应=eps_min'
assert abs(eps_linear(500, 1000) - 0.525) < 1e-3, '中点应在 eps0,eps_min 之间'
# 单调不增
lin = [eps_linear(e, 1000) for e in range(0, 1001, 100)]
assert all(lin[i] >= lin[i+1] - 1e-12 for i in range(len(lin)-1)), 'ε 应单调不增'
# 指数: 第0回合=eps0，永远≥eps_min
assert abs(eps_exponential(0) - 1.0) < 1e-9
assert eps_exponential(100000) == 0.05, '指数衰减应触底在 eps_min'
print('✅ 练习 3 通过：两种 ε 衰减调度正确(早探索晚利用、留底)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def td_error(V, s, r, sp, done, gamma):
    return r + (0.0 if done else gamma * V[sp]) - V[s]

def td_update(V, s, delta, alpha):
    V[s] += alpha * delta
    return V

In [ ]:
# 练习 2 参考答案
def q_target(Q, r, spi, done, gamma):
    return r + (0.0 if done else gamma * Q[spi].max())

def sarsa_target(Q, r, spi, ap, done, gamma):
    return r + (0.0 if done else gamma * Q[spi, ap])

In [ ]:
# 练习 3 参考答案
def eps_linear(ep, n_eps, eps0=1.0, eps_min=0.05):
    frac = ep / n_eps
    return max(eps_min, eps0 + frac * (eps_min - eps0))

def eps_exponential(ep, decay=0.999, eps0=1.0, eps_min=0.05):
    return max(eps_min, eps0 * decay ** ep)

---
## 🧪 真实数据胶囊：Robbins-Monro 步长与收敛

TD/Q-learning 的收敛靠 Robbins-Monro 步长条件 $\sum\alpha_t=\infty,\ \sum\alpha_t^2<\infty$。

用一个**真实**的随机近似任务验证：估计带噪样本的均值（这正是 TD 在单状态下的退化形式）。样本 $x_t \sim \mathcal{N}(\mu, \sigma^2)$，增量更新 $\hat\mu \leftarrow \hat\mu + \alpha_t(x_t - \hat\mu)$。比较三种步长：$1/t$(满足RM)、常数(不满足第二条)、$1/\sqrt{t}$(不满足第二条)。

In [ ]:
def stochastic_mean(alpha_schedule, mu=3.0, sigma=2.0, n=20000, seed=0):
    rng = np.random.default_rng(seed)
    est = 0.0; traj = []
    for t in range(1, n + 1):
        x = rng.normal(mu, sigma)
        alpha = alpha_schedule(t)
        est += alpha * (x - est)        # 随机近似更新(= 单状态 TD)
        traj.append(est)
    return np.array(traj)

mu = 3.0
traj_rm   = stochastic_mean(lambda t: 1.0 / t)            # 1/t：满足 RM
traj_const = stochastic_mean(lambda t: 0.01)             # 常数：不满足 Σα²<∞
for name, traj in [('1/t (RM)', traj_rm), ('常数 0.01', traj_const)]:
    final_err = abs(traj[-1] - mu)
    late_std = traj[-2000:].std()           # 后期抖动幅度
    print(f'{name:12s}: 末值={traj[-1]:.3f} 误差={final_err:.3f} 后期抖动std={late_std:.4f}')
# 1/t 步长：抖动趋于 0(收敛到点)；常数步长：持续抖动(不收敛到点，在 μ 附近游走)
assert traj_rm[-2000:].std() < traj_const[-2000:].std(), '1/t 后期抖动应远小于常数步长'
assert abs(traj_rm[-1] - mu) < 0.1, '1/t 步长应收敛到真均值'
print('\n关键：1/t(满足RM)收敛到点；常数步长在真值附近持续抖动(不满足 Σα²<∞)')
print('✅ 验证 Robbins-Monro：这正是 TD/Q-learning 收敛性的微观机制')

**🧪 胶囊练习**：实现 `satisfies_rm(alphas)`：给定一串步长，**数值近似**判断它是否满足 Robbins-Monro 两条件。返回 `(sum_diverges, sq_sum_converges)` 两个布尔（用足够大的部分和近似：sum 很大视为发散迹象，平方和增长趋缓视为收敛迹象）。提示：对 `1/t` 序列，`Σα` 随 log 增长(发散)、`Σα²` 收敛到 π²/6。

In [ ]:
def satisfies_rm(alphas):
    # TODO: s1=sum(alphas), s2=sum(a*a for a in alphas)
    #       sum_diverges: s1 是否很大(>10，作为发散的近似迹象)
    #       sq_sum_converges: s2 是否有界(<10，作为收敛的近似迹象)
    #       返回 (sum_diverges, sq_sum_converges)
    raise NotImplementedError

In [ ]:
# 自测
alphas_rm = [1.0 / t for t in range(1, 100000)]         # 1/t
alphas_const = [0.1] * 100000                          # 常数
div_rm, conv_rm = satisfies_rm(alphas_rm)
div_c, conv_c = satisfies_rm(alphas_const)
assert div_rm and conv_rm, '1/t 应满足两条件(和发散、平方和收敛)'
assert div_c and not conv_c, '常数步长: 和发散但平方和也发散(不满足第二条)'
print('1/t      :', (div_rm, conv_rm), '-> 满足 RM，收敛到点')
print('常数 0.1 :', (div_c, conv_c), '-> 平方和发散，不收敛到点(持续抖动)')
print('✅ 胶囊练习通过：能判别步长是否满足 Robbins-Monro')

In [ ]:
# 📖 胶囊参考答案
def satisfies_rm(alphas):
    s1 = sum(alphas)
    s2 = sum(a * a for a in alphas)
    return (s1 > 10.0, s2 < 10.0)

### 小结
- **TD** = 自举(像DP) + 采样(像MC)：走一步就学，无需模型、无需等回合结束。
- **TD 误差** δ = r + γV(s') − V(s) = 惊喜程度，是贯穿全 RL 的通用信号。
- **Q-learning**(off-policy)：目标用 **max** → 学 Q*，可从任意数据学(→DQN)。
- **SARSA**(on-policy)：目标用**实际动作** → 学当前策略，把探索风险内化(Cliff 走安全路)。
- **收敛**：Robbins-Monro 步长 + 充分探索 → 表格型以概率 1 收敛(函数逼近则有 deadly triad)。

下一站：**模块 03 · 策略梯度与 REINFORCE** —— 不再学价值再贪心，而是直接对策略求梯度。